# VoiceForge — Talking-head VIDEO backend

Turns one photo + your generated narration into a lip-synced MP4, using **SadTalker** (lip sync + blinking + head motion — best for short clips) and **Wav2Lip** (mouth-only, fast enough for long narration). Exposes a public HTTPS API via a Cloudflare tunnel.

**This is a SEPARATE backend from the TTS notebook.** These models need numpy 1.x, which conflicts with the TTS session — so run this in its own Colab session, after your speech is done, so the single free GPU is reused rather than needed twice at once.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save.

**The sequence — same rule as the TTS notebook:** click **Run all**. Cell 1 installs both engines (~8–12 min the first time) and then tells you to **restart the session**. Restart, then click **Run all again from the top**. The numpy pin only takes effect in a fresh kernel; Cell 2 asserts it and will tell you plainly if it didn't stick.

Cell 3 prints the backend URL — paste it into the VoiceForge **Video** tab. Keep this tab open while rendering.

In [ ]:
# Cell 1 — Install (SadTalker + Wav2Lip). ~8–12 min the first time.
import os

# NOTE: Colab runs Python 3.12. torch 2.0.1 / torchvision 0.15.2 / numpy 1.23.5
# have NO 3.12 wheels — pinning them silently fails and leaves numpy 2.x, which
# breaks SadTalker (np.VisibleDeprecationWarning was removed in numpy 2).
# So: use Colab's own torch, and pin numpy to 1.26.4 — the newest numpy that is
# both <2 (keeps the numpy-1 API these repos need) and 3.12-compatible.

!pip install -q fastapi uvicorn nest-asyncio python-multipart soundfile imageio imageio-ffmpeg
!pip install -q facexlib gfpgan basicsr insightface onnxruntime moviepy \
  opencv-python-headless yacs kornia pydub librosa numba resampy \
  scikit-image tqdm safetensors av face-alignment
# batch_face = the RetinaFace detector the Wav2Lip fork imports.
!pip install -q batch-face

# --- SadTalker + checkpoints
if not os.path.exists('/content/SadTalker'):
    !git clone -q https://github.com/OpenTalker/SadTalker /content/SadTalker
!cd /content/SadTalker && bash scripts/download_models.sh

# --- Wav2Lip (justinjohn0306 fork: newer-torch friendly + hosts checkpoints)
if not os.path.exists('/content/Wav2Lip'):
    !git clone -q https://github.com/justinjohn0306/Wav2Lip /content/Wav2Lip
!mkdir -p /content/Wav2Lip/checkpoints /content/Wav2Lip/face_detection/detection/sfd

# The fork's inference.py hardcodes checkpoints/mobilenet.pth for its RetinaFace
# detector, so grab every checkpoint the release ships — missing one shows up
# later as a confusing FileNotFoundError mid-render.
W2L_REL = 'https://github.com/justinjohn0306/Wav2Lip/releases/download/models'
for name in ['wav2lip.pth', 'wav2lip_gan.pth', 'mobilenet.pth', 'resnet50.pth']:
    !wget -q --show-progress -O /content/Wav2Lip/checkpoints/{name} {W2L_REL}/{name}
!wget -q -O /content/Wav2Lip/face_detection/detection/sfd/s3fd.pth {W2L_REL}/s3fd.pth

# A 404 leaves a 0-byte file, which fails much later and cryptically. Check now.
print('\nWav2Lip checkpoint sizes:')
for p in ['checkpoints/wav2lip.pth', 'checkpoints/wav2lip_gan.pth',
          'checkpoints/mobilenet.pth', 'checkpoints/resnet50.pth',
          'face_detection/detection/sfd/s3fd.pth']:
    full = f'/content/Wav2Lip/{p}'
    mb = os.path.getsize(full) / 1e6 if os.path.exists(full) else 0
    print(f'  {"OK " if mb > 1 else "BAD"} {p}: {mb:.1f} MB')

# --- Pin numpy LAST so nothing above can drag it back to 2.x (same lesson as
# the TTS notebook). Not -q: if this fails we need to SEE it.
!pip install --force-reinstall "numpy==1.26.4"

# --- Newer torchvision removed transforms.functional_tensor, which basicsr and
# facexlib still import. Rewrite those imports in place.
!grep -rl "torchvision.transforms.functional_tensor" /usr/local/lib/python3.12/dist-packages/ 2>/dev/null \
  | xargs -r sed -i 's/torchvision\.transforms\.functional_tensor/torchvision.transforms.functional/g'

# --- Cloudflare tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

print('=' * 60)
print('Installed. RESTART THE SESSION NOW (Runtime > Restart session),')
print('then Run All again from the top. The numpy downgrade only takes')
print('effect in a fresh kernel.')
print('=' * 60)

In [ ]:
# Cell 2 — Verify GPU + environment + which engines are ready
import torch, numpy as np, os, glob
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU.'
GPU = torch.cuda.get_device_name(0)
print(GPU, '| torch', torch.__version__, '| numpy', np.__version__)

# SadTalker needs the numpy-1 API (np.VisibleDeprecationWarning etc.). If this
# trips, Cell 1's pin didn't stick — re-run Cell 1 AND restart the session.
assert np.__version__.startswith('1.'), (
    f'numpy is {np.__version__}, need 1.x. Re-run Cell 1, then RESTART the '
    'session (Runtime > Restart session) and Run All again.')

def _have(path, min_mb=1):
    return os.path.exists(path) and os.path.getsize(path) / 1e6 >= min_mb

# checkpoints present is enough; gfpgan weights auto-download on first run
SAD_OK = bool(glob.glob('/content/SadTalker/checkpoints/*'))

# Every file the fork's inference.py touches — mobilenet.pth is the RetinaFace
# detector it hardcodes, and its absence only shows up mid-render otherwise.
W2L_FILES = [
    '/content/Wav2Lip/checkpoints/wav2lip_gan.pth',
    '/content/Wav2Lip/checkpoints/mobilenet.pth',
    '/content/Wav2Lip/face_detection/detection/sfd/s3fd.pth',
]
W2L_OK = all(_have(p) for p in W2L_FILES)

print('sadtalker ready:', SAD_OK)
print('wav2lip ready:  ', W2L_OK)
if not W2L_OK:
    for p in W2L_FILES:
        if not _have(p):
            print('   missing/empty:', p)
if not (SAD_OK or W2L_OK):
    print('Neither engine set up — re-run Cell 1 and watch for download errors.')

In [ ]:
# Cell 3 — API server + public URL
import io, os, base64, subprocess, threading, time, re, glob, shutil, uuid
import soundfile as sf, torch, nest_asyncio, uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'],
                   allow_methods=['*'], allow_headers=['*'])

WORK = '/content/vf_video'; os.makedirs(WORK, exist_ok=True)

def _engines():
    e = []
    if SAD_OK: e.append('sadtalker')
    if W2L_OK: e.append('wav2lip')
    return e

def _run(cmd, cwd):
    p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError((p.stderr or p.stdout or 'unknown error')[-1600:])

def to_wav(src_path, dst_path):
    """Both engines need real 16k mono WAV. The app can send mp3/m4a/webm
    (uploaded audio) as well as our own WAV, so always transcode."""
    p = subprocess.run(
        ['ffmpeg', '-y', '-i', src_path, '-ac', '1', '-ar', '16000', dst_path],
        capture_output=True, text=True)
    if p.returncode != 0 or not os.path.exists(dst_path):
        raise RuntimeError('Could not decode the audio: ' + (p.stderr or '')[-800:])
    return dst_path

def run_sadtalker(image_path, audio_path, out_dir):
    # --still reduces head jitter; --preprocess full keeps the whole photo
    # framed (vs 'crop' which zooms to the face); gfpgan sharpens the result.
    _run(['python', 'inference.py', '--driven_audio', audio_path,
          '--source_image', image_path, '--result_dir', out_dir,
          '--still', '--preprocess', 'full', '--enhancer', 'gfpgan'],
         cwd='/content/SadTalker')
    mp4s = sorted(glob.glob(os.path.join(out_dir, '**', '*.mp4'), recursive=True),
                  key=os.path.getmtime)
    if not mp4s: raise RuntimeError('SadTalker produced no mp4')
    return mp4s[-1]

def run_wav2lip(image_path, audio_path, out_path):
    _run(['python', 'inference.py', '--checkpoint_path', 'checkpoints/wav2lip_gan.pth',
          '--face', image_path, '--audio', audio_path, '--outfile', out_path,
          '--nosmooth', '--pads', '0', '15', '0', '0'],
         cwd='/content/Wav2Lip')
    if not os.path.exists(out_path): raise RuntimeError('Wav2Lip produced no mp4')
    return out_path

@app.get('/health')
def health():
    return {'ok': True, 'gpu': GPU, 'engines': _engines()}

@app.post('/animate')
async def animate(payload: dict):
    engine = payload.get('engine', 'sadtalker')
    if engine not in _engines():
        return JSONResponse({'error': f'{engine} not loaded'}, status_code=400)
    job = os.path.join(WORK, uuid.uuid4().hex[:10]); os.makedirs(job, exist_ok=True)
    try:
        mime = payload.get('image_mime', '')
        ext = 'jpg' if ('jpeg' in mime or 'jpg' in mime) else 'png'
        img = os.path.join(job, f'face.{ext}')
        raw = os.path.join(job, 'audio_raw')
        wav = os.path.join(job, 'audio.wav')
        with open(img, 'wb') as f: f.write(base64.b64decode(payload['image_b64']))
        with open(raw, 'wb') as f: f.write(base64.b64decode(payload['audio_b64']))
        to_wav(raw, wav)
        info = sf.info(wav); dur = info.frames / info.samplerate
        t0 = time.time()
        if engine == 'sadtalker':
            mp4 = run_sadtalker(img, wav, os.path.join(job, 'out'))
        else:
            mp4 = run_wav2lip(img, wav, os.path.join(job, 'out.mp4'))
        with open(mp4, 'rb') as f:
            b64 = base64.b64encode(f.read()).decode()
        return {'video_b64': b64, 'duration': round(dur, 2),
                'gen_seconds': round(time.time() - t0, 2)}
    except Exception as e:
        return JSONResponse({'error': str(e)[-1600:]}, status_code=500)
    finally:
        shutil.rmtree(job, ignore_errors=True)
        torch.cuda.empty_cache()

# --- start server + tunnel ---
nest_asyncio.apply()
threading.Thread(
    target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000, log_level='error'),
    daemon=True,
).start()
time.sleep(3)

proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    m = re.search(r'https://[-\w]+\.trycloudflare\.com', line)
    if m:
        print('\n' + '=' * 60)
        print('  VIDEO BACKEND URL — paste into the VoiceForge Video tab:')
        print('  ' + m.group(0))
        print('=' * 60 + '\n')
        print('Engines available:', _engines())
        print('Keep this tab open while rendering.')
        break

In [ ]:
# Cell 4 — Keep-alive (optional)
# Colab disconnects idle tabs. Run this in the background during long renders.
import time
while True:
    time.sleep(60)
    print('.', end='', flush=True)